# Data Visualization Project: Colorado 14ers
**Author:** Annalena Salchegger <br>
**Topic:** Analysis of Colorado's Fourteeners<br><br>

This notebook explores a dataset of Colorado’s 14ers (mountains above 14,000 ft), focusing on their geospatial distribution, hiking difficulty, and visitor patterns. <br>
Using Pandas for preprocessing and Altair for interactive visualizations, I investigate three main research questions and present insights backed by data. <br><br>
Beyond the data analysis, the project also emphasizes accessibility: a simple website that presents the research questions with interactive visualizations in a user-friendly manner, including a screen reader option for blind or visually impaired users. This combines my passion for hiking and the outdoors with my commitment to designing technology that is fair, inclusive and non-discriminatory.

In [1]:
import pandas as pd
import altair as alt

In [2]:
#importing the data from a CSV file and reading it into a DataFrame
#the 'Standard Route' column has weird characters (e.g. Northeast Ridge� ) meaning the file is not UTF-8 encoded, 'ISO-8859-1' fixes the problem

colorado14er = pd.read_csv(r'14er.csv', encoding='ISO-8859-1') 
df = pd.DataFrame(colorado14er)

colorado14er.head()

,ID,Mountain Peak,Mountain Range,Elevation_ft,fourteener,Prominence_ft,Isolation_mi,Lat,Long,Standard Route,Distance_mi,Elevation Gain_ft,Difficulty,Traffic Low,Traffic High,photo
0,1,Mount Elbert,Sawatch Range,14440,Y,9093,670.00,39.1178,-106.4454,Northeast Ridge,9.50,4700,Class 1,20000,25000,https://www.14ers.com/photos/mtelbert/peakphot...
1,2,Mount Massive,Sawatch Range,14428,Y,1961,5.06,39.1875,-106.4757,East Slopes,14.50,4500,Class 2,7000,10000,https://www.14ers.com/photos/mtmassive/peakpho...
2,3,Mount Harvard,Sawatch Range,14421,Y,2360,14.93,38.9244,-106.3207,South Slopes,14.00,4600,Class 2,5000,7000,https://www.14ers.com/photos/harvardgroup/peak...
3,4,Blanca Peak,Sangre de Cristo Range,14351,Y,5326,103.40,37.5775,-105.4856,Northwest Ridge,17.00,6500,Hard Class 2,1000,3000,https://www.14ers.com/photos/blancagroup/peakp...
4,5,La Plata Peak,Sawatch Range,14343,Y,1836,6.28,39.0294,-106.4729,Northwest Ridge,9.25,4500,Class 2,5000,7000,https://www.14ers.com/photos/laplatapeak/peakp...


In [3]:
colorado14er.info()

print(f"\nTotal data items: {colorado14er.size}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 58 non-null     int64  
 1   Mountain Peak      58 non-null     object 
 2   Mountain Range     58 non-null     object 
 3   Elevation_ft       58 non-null     int64  
 4   fourteener         58 non-null     object 
 5   Prominence_ft      58 non-null     int64  
 6   Isolation_mi       58 non-null     float64
 7   Lat                58 non-null     float64
 8   Long               58 non-null     float64
 9   Standard Route     58 non-null     object 
 10  Distance_mi        58 non-null     float64
 11  Elevation Gain_ft  58 non-null     int64  
 12  Difficulty         58 non-null     object 
 13  Traffic Low        58 non-null     int64  
 14  Traffic High       58 non-null     int64  
 15  photo              58 non-null     object 
dtypes: float64(4), int64(6), obj

## Dataset Overview

**Description:** 
<br> The dataset contains information about 58 distinct mountain peaks in the state of Colorado, USA, which are known as "fourteeners" - meaning that they rise above 14,000 feet above sea level. Some of the key features of the dataset are: elevation, prominence, route difficulty, estimated visitor traffic as well as latitude and longitude.
<br>

**Source:** 
<br>I downloaded the dataset from Kaggle. <br>
https://www.kaggle.com/datasets/mikeshout/14erpeaks?resource=download
<br>

**Original Dataset size:**
- Rows: 58 total 
- Columns: 16
- Data items: 928 total

**Relevant Columns and Rows for this project:**
- Relevant Columns: Mountain Peak, Mountain Range, Elevation_ft, Prominence_ft, Lat, Long, Distance_mi, Elevation Gain_ft, Difficulty, Traffic Low, Traffic High
- Dropped Columns: ID, fourteener, Standard Route, Isolation_mi, photo
- Relevant Rows: all containing a 'Y' in the column fourteener (all mountains which are officially considered 14ers by the dataset)
- Irrelevant Rows: all containing a 'N' in the column fourteener (last 5 rows)

*<u>Reasoning:</u> To be ranked, a peak must have at least 300 feet of prominence, which is the amount of elevation it rises above the lowest saddle that connects to the nearest, higher peak. This guideline has been in use in Colorado for some time. The following peaks are not ranked because they do not have enough prominence but are on this 14er list because they are named and recognized on USGS maps:*

| Mountain Peak|  Elevation_ft |   Prominence_ft |
|:-----:|:-----:|:-----:|
|  Mt. Cameron     | 	 14,248 | 152| 
| El Diente Peak  | 	 14,175| 264 |
| Challenger Point| 	 14,086 | 264|
| North Eolus      |	 14,042|  212 |
| Conundrum Peak   |	 14,037|  225| 

Information taken from: https://www.14ers.com/14ers


## 2. Research Questions for Analysis

**Focus:** Understanding spatial, quantitative and categorical patterns in Colorado's 14ers.
1. **Geospatial and Semantic Focus:** How are Colorado's 14ers distributed geographically across different mountain ranges, and does elevation vary by region?
2. How does elevation gain on standard routes correlate with distance, difficulty and visitor traffic? Are longer routes necessarily harder and less visited? 
3. Which mountain range has the highest average elevation, and how do other ranges compare?

## 3. Data Modifications

**In order to answer my research questions, some data modifications are helpful. Here is a list of all the transformations I will apply to the dataset, including the reasoning behind it.**

1. The column 'fourteener' contains 5 peaks that are officially not considered a fourteener by the dataset *(fourteener == 'N')*, therefore I filtered these rows out (step 2) and won't consider them.

2. I think in addition to having a High and Low estimate of the traffic, it makes sense to create a new column with the mean of both values <br>
    *(Traffic Low + Traffic High) / 2*
<br>

3. The Difficulty is denoted with a class label from 1 to 5. Some entries contain an additional adjective (e.g. 'hard', 'easy'), which I will get rid of, because it does not change the class and helps simplifying the visualizations later on.

## 4. Data Cleaning and Preprocessing

**Objectives:**
- Removing unnecessary columns
- Filtering out non-official 14ers
- Creating a new column `Average Visitors 2017` as the mean of `Traffic Low` and `Traffic High`
- Cleaning the `Difficulty` column by removing adjectives like "Hard" or "Easy" for simplicity

In [4]:
#dropping unnecessary columns
df_new = colorado14er.drop(columns=['ID', 'Standard Route', 'Isolation_mi', 'photo'])

In [5]:
#filtering out only the officially documented 14ers, and then dropping the 'fourteener' column because it is not needed anymore
df_col14er = df_new[df_new['fourteener'] == 'Y'].drop(columns=['fourteener'])
df_col14er.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53 entries, 0 to 52
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Mountain Peak      53 non-null     object 
 1   Mountain Range     53 non-null     object 
 2   Elevation_ft       53 non-null     int64  
 3   Prominence_ft      53 non-null     int64  
 4   Lat                53 non-null     float64
 5   Long               53 non-null     float64
 6   Distance_mi        53 non-null     float64
 7   Elevation Gain_ft  53 non-null     int64  
 8   Difficulty         53 non-null     object 
 9   Traffic Low        53 non-null     int64  
 10  Traffic High       53 non-null     int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 5.0+ KB


In [6]:
# adding a new column 'Average Visitors 2017' to the dataframe
df_col14er['Average Visitors 2017'] =((df_col14er['Traffic Low'] + df_col14er['Traffic High']) / 2).astype(int)
df_col14er[['Mountain Peak', 'Traffic Low', 'Traffic High', 'Average Visitors 2017']].head()

,Mountain Peak,Traffic Low,Traffic High,Average Visitors 2017
0,Mount Elbert,20000,25000,22500
1,Mount Massive,7000,10000,8500
2,Mount Harvard,5000,7000,6000
3,Blanca Peak,1000,3000,2000
4,La Plata Peak,5000,7000,6000


In [7]:
# getting rid of hard and easy in the 'Difficulty' column, using a regular expression
df_col14er['Difficulty'] = df_col14er['Difficulty'].str.replace(r'\b(Hard|Easy)\s+', '', regex=True)
df_col14er[['Mountain Peak', 'Difficulty']].head()

,Mountain Peak,Difficulty
0,Mount Elbert,Class 1
1,Mount Massive,Class 2
2,Mount Harvard,Class 2
3,Blanca Peak,Class 2
4,La Plata Peak,Class 2


## 5. Analytical Tasks

**Derived from my research questions:**

**Question 1:**<br>
   *"How are Colorado's 14ers distributed geographically across different mountain ranges, and does elevation vary by region?"* <br>
- Clustering/Grouping: Identify how peaks are grouped by mountain range
- Spatial Localization: Detect where 14ers are located across the geographic map
- Comparison: Compare elevation values across regions (e.g., which areas have higher or lower peaks, using a color ramp)
- Association: Connect elevation values with specific geographic regions or ranges
- Trend Detection: Observe elevation patterns across longitude and latitude
<br><br>

**Question 2:** <br>
   *"How does elevation gain on standard routes correlate with distance, difficulty and visitor traffic? Are longer routes necessarily harder and less visited?*"<br>
- Correlation: Identify relationships between elevation gain, distance, difficulty, and visitor traffic
- Comparison: Compare hikes across multiple dimensions (elevation, distance, difficulty, visitor count)
- Outlier Detection: Spot unusual hikes (e.g., very long but easy hikes, or short but hard ones)
- Ranking: Determine which hikes are most or least visited
- Pattern Recognition: See if there are general trends (e.g., harder hikes = fewer visitors)
<br><br>

**Question 3:** <br>
    *"Which mountain range has the highest average elevation, and how do other ranges compare?"* <br>
- Aggregation: Calculate and interpret average elevation per range
- Comparison: Compare the average elevations between mountain ranges
- Ranking: Identify which range ranks highest in terms of elevation
- Classification: Group peaks by mountain range to compare aggregated values
- Outlier Detection: Spot which ranges are significantly higher or lower than the rest


## 6. Data Attributes

**Goal:** Identifying which data attributes are important for answering my research questions. <br>
**Reason:** Specifying the number of categories contained in each ordinal and nominal data column, as well as the range for quantitative columns gives a great overview and helps with understanding the data.

| Attribute |  Type |   Additional Specification |
|:-----:|:-----:|:-----:|
|  Mountain Peak  | Nominal     | 53 unique values  | 
| Mountain Range  | Nominal     | 6 distinct values |
| Elevation_ft    | Quantitative| Value Range: 14007 - 14440 ft |
| Prominence_ft   |Quantitative | Value Range: 301 - 9093 ft       |
| Lat             |Quantitative (geospatial) | Southernmost Peak: 37.122400 - Northernmost Peak: 40.255000  | 
| Long            |Quantitative (geospatial) | Westernmost Peak: -107.991600 - Easternmost Peak: -105.044200| 
| Difficulty      |Ordinal      | Classes 1-5 (1=easiest, 5=most difficult)       | 
| Distance_mi     |Quantitative | Value Range: 3.25 - 26.00 mi |
| Elevation Gain_ft|Quantitative| Value Range: 2000 - 7500 ft   |	
| Traffic Low     |Quantitative | Min 1000 visitors per year       | 
| Traffic High    |Quantitative | Max 40000 visitors per year        | 

** *Without considering the 5 mountains which don't belong to the official fourteeners.*


**Semantic Structure:** Geospatial (Lat/Long) and hierarchical (peaks grouped by mountain range).

In [8]:
#for categorical data
df_col14er.describe(include=['object'])

,Mountain Peak,Mountain Range,Difficulty
count,53,53,53
unique,53,6,4
top,Mount Elbert,Sawatch Range,Class 2
freq,1,15,31


In [9]:
#for quantitative data
#rounding Lat and Long to 4 decimal places, all others to two - for better readability
desc = df_col14er.describe()
desc['Lat'] = desc['Lat'].round(4)
desc['Long'] = desc['Long'].round(4)
desc.loc[:, desc.columns.difference(['Lat', 'Long'])] = desc.loc[:, desc.columns.difference(['Lat', 'Long'])].round(2)
desc

,Elevation_ft,Prominence_ft,Lat,Long,Distance_mi,Elevation Gain_ft,Traffic Low,Traffic High,Average Visitors 2017
count,53.00,53.00,53.0000,53.0000,53.00,53.00,53.00,53.00,53.00
mean,14163.60,2018.49,38.5914,-106.4220,11.09,4403.77,6566.04,9264.15,7915.09
std,120.47,1669.66,0.7348,0.7842,4.70,1331.38,7811.50,8981.07,8392.84
min,14007.00,301.00,37.1224,-107.9916,3.25,2000.00,1000.00,3000.00,2000.00
25%,14058.00,850.00,37.9647,-106.9890,7.50,3300.00,1000.00,3000.00,2000.00
50%,14158.00,1635.00,38.8405,-106.3138,11.00,4500.00,3000.00,5000.00,4000.00
75%,14271.00,2503.00,39.1188,-105.6688,14.00,5400.00,5000.00,7000.00,6000.00
max,14440.00,9093.00,40.2550,-105.0442,26.00,7500.00,35000.00,40000.00,37500.00


## 7. Visual Analysis

### **Question 1:**
**Geospatial distribution and elevation by region**

* **Charts:** Scatter plot, Map overlay
* **Insights:** Elevation clusters, geographic trends, range clustering

In [10]:
mountain_range = df_col14er['Mountain Range'].unique() # get unique field values
mountain_range = list(filter(lambda d: d is not None, mountain_range)) # filter out None values
mountain_range.sort()

In [11]:
#Question 1 | Chart 1: Scatter Plot with Selection Dropdown

#dropdown options with "None"
ranges_with_all = ['None'] + mountain_range

#creating a selection dropdown
select_range = alt.selection_point(
    name='Select',
    fields=['Mountain Range'],
    bind=alt.binding_select(options=ranges_with_all),
    value='None' 
)

condicont = alt.condition(select_range, alt.value(0.9), alt.value(0.1))


scatter_1 = alt.Chart(df_col14er).mark_point(size=100, opacity=0.7).encode(
    alt.X('Long:Q', title='Longitude', scale=alt.Scale(domain=[-108.5, -104.5])),
    alt.Y('Lat:Q', title='Latitude', scale=alt.Scale(domain=[36.5, 40.5])),
    alt.Color('Elevation_ft:Q', title='Elevation (ft)', scale=alt.Scale(scheme='plasma')),
    tooltip=['Mountain Peak', 'Elevation_ft'],
    opacity=condicont
).properties(
    width=600,
    height=500,
    title={
        "text": "Geospatial Distribution of Colorado 14ers and their Elevation",
        "subtitle": ["Use the Selection field below to highlight a specific Mountain Range", ""]
    }
).add_params(
    select_range
)

scatter_1

alt.Chart(...)

In [12]:
#Question 1 | Chart 2: Map Overlay with Selection Dropdown

#Due to the limited support of altair features, visualizing the map of Colorado with the topography or major cities is not possible.
#Therefore, the map is kept simple with a grey background and white borders.

from vega_datasets import data

ranges_with_all = ['None'] + mountain_range

select_range = alt.selection_point(
    name='Select',
    fields=['Mountain Range'],
    bind=alt.binding_select(options=ranges_with_all),
    value='None'  # default to showing none
)

#load the US states data from Vega datasets
states = alt.topo_feature(data.us_10m.url, 'states')
colorado = alt.Chart(states).transform_filter(alt.datum.id == 8)

#create a base map
background = colorado.mark_geoshape(
    fill='gainsboro',
    stroke='white'
).project('albersUsa').properties(
    width=700,
    height=500)

points = alt.Chart(df_col14er).mark_circle(size=100).encode(
    longitude='Long:Q',
    latitude='Lat:Q',
    color=alt.Color('Elevation_ft:Q', scale=alt.Scale(scheme='plasma')),
    tooltip=['Mountain Peak:N', 'Elevation_ft:Q'],
    opacity = condicont
).add_params(
    select_range
)

#combine the map and the data points
map = (background + points).properties(
    title={
        "text": "Geospatial Distribution of Colorado 14ers and their Elevation, Colorado Map",
        "subtitle": ["Use the Selection field below to highlight a specific Mountain Range", ""]
    }
)

map

alt.LayerChart(...)

In [13]:
#Combinig the charts for comparison
combined_chart1 = alt.hconcat(
    scatter_1,
    map
).configure_concat(
    spacing=50  #space between the charts
).properties(
    title={
        "text": "How are Colorado's 14ers distributed geographically across different mountain ranges, and does elevation vary by region?",
        "subtitle": ["For additional information hover over the points", "", ""],  # adding an empty string for padding
        "fontSize": 18,
        "subtitleFontSize": 14,
        "anchor": 'middle'
    }
)

combined_chart1

alt.HConcatChart(...)

#### **a) Visual Encodings and Interactivity**

**Marks:**
- Scatter Plot ('Geospatial Distribution of Colorado 14ers and their Elevation'): mark_points, points represent individual mountain peaks
- Map Overlay ('Geospatial Distribution of Colorado 14ers and their Elevation, Colorado Map'): mark_circle, circles represent individual mountain peaks

**Encoding Channels** (identical for both plots):
- Magnitude Channels (Quantitative):
    - X: Long (= Longitude), position on a common scale
    - Y: Lat (= Latitude), position on a common scale
- Identity Channels (Nominal):
    - Color: Elevation_ft, color gradient to indicate elevation (scheme = "plasma")

**Interactivity:**
- Mountain Range selection, to highlight the peaks of a specific mountain range
- Tooltip that displays Mountain Peak (nominal) and Elevation_ft (quantitative) on hover

#### **b) Strongest Visualization and Rationale**

**Scatter Plot 1:**
This scatter plot maps the geographic coordinates with longitude on the x-axis and latitude on the y-axis. Each point represents the location of a mountain, plotted precisely according to its position on the globe. The result is a spatial distribution of the mountain peaks, offering a visual sense of their spread across the different regions. For better regional understanding, the Select_Mountain _Range button at the bottom, highlights individually chosen Mountain Ranges for you.

**Scatter Plot 2:**
Scatter plot 2 is best for understanding how the 14ers are distributed across Colorado, given the outlines of the state in the background. Each circle represents one mountain peak and the range can be chosen via the select button under the plot. This feature makes it easy to visually spot the spatial distributions between ranges and elevations of the individual peaks. Unfortunately, I was not able to include landmarks or the topography in the map of Colorado which makes it hard to grasp the meaning and distribution of the dots, especially for people who don't have sufficient knowledge about the geography of Colorado.

**Final Conclusion:** Scatter Plot 1 is more effective for answering the original question: *"How are Colorado's 14ers distributed geographically across different mountain ranges, and does elevation vary by region?"*
It clearly separates the mountain ranges spatially using their location which can be selected manually under the plot. In addition to that the elevation and name of each mountain peak is provided via the tooltip. This chart answers both parts of the question — distribution and regional grouping — more directly with a clear and structured background.

#### **c) Gained insights**

- **Elevation Clusters:** Highest elevations (yellow) are generally found in the central and southwest regions of the state
- **Geospatial Trends:** 14ers are not randomly distributed - they follow a clear north-south 'mountainous' corridor
- **Geographic Spread:** Very few peaks appear in the eastern part of the state, aligning with Colorado’s topography
- **Mountain Range Clustering:** Peaks are grouped in distinct geographic clusters according to their range
- **Elevation by Region:** Some ranges (like the San Juan Range) may have many peaks, but not necessarily the tallest ones
- **Rocky Mountains:** Most ranges are located along a central vertical axis, consistent with the Rocky Mountains cutting through the state

### **Question 2**: <br>
**Elevation gain vs distance, difficulty and visitors**

* **Charts:** Scatter plot, Faceted bubble chart
* **Insights:** Correlation between distance/elevation, visitor distribution by difficulty

In [14]:
#Question 2 | Chart 1: Scatter plot with interactive features

scatter_2 = alt.Chart(df_col14er).mark_circle(opacity=0.7).encode(
    x=alt.X('Distance_mi:Q', title='Route Distance (miles)', scale=alt.Scale(domain=[0, 28]), axis=alt.Axis(labelFontSize=12)),
    y=alt.Y('Elevation Gain_ft:Q', title='Elevation Gain (feet)', axis=alt.Axis(labelFontSize=12)),
    color=alt.Color('Difficulty:N').scale(scheme='blueorange'), #plasma, #magma
    size=alt.Size('Average Visitors 2017:Q', scale=alt.Scale(range=[0,800])),
    tooltip=['Mountain Peak', 'Difficulty', 'Distance_mi', 'Elevation Gain_ft', 'Average Visitors 2017'] # nice addition to show more information when hovering over the points
).properties(
    height=300,
    width=600,
    title={
        "text" : 'Route Distance vs Elevation Gain by Difficulty and Average Visitors',
        "subtitle": "For more information zoom in or hover over the points",
        "fontSize": 16,
        "fontWeight": "bold",
        "subtitleFontSize": 12
    }
).interactive()

scatter_2

alt.Chart(...)

In [15]:
#Question 2 | Chart 2: Bubble chart faceted by Difficulty

bubble = alt.Chart(df_col14er).mark_circle(opacity=0.7).encode(
    x=alt.X('Distance_mi:Q', title='Route Distance (miles)', axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    y=alt.Y('Elevation Gain_ft:Q', title='Elevation Gain (feet)', axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    color=alt.Color('Difficulty:N', scale=alt.Scale(scheme='blueorange')),
    size=alt.Size('Average Visitors 2017:Q', scale=alt.Scale(range=[0, 800])),
    tooltip=['Mountain Peak', 'Difficulty', 'Distance_mi', 'Elevation Gain_ft', 'Average Visitors 2017']
).properties(
    width=300,
    height=300,
)

facet_bubble = bubble.facet(
    column=alt.Column('Difficulty:N', title='Difficulty')
).properties(
    title={
        "text": 'Route Distance vs Elevation Gain by Difficulty and Average Visitors',
        "fontSize": 16,
        "fontWeight": "bold",
        "subtitle": ["For more information zoom in or hover over the points", ""],  # empty string for extra space
        "anchor": "middle"
    }
).interactive()

facet_bubble

## NOTE: I wanted to layer the individual charts in a 2x2 grid, but alt.hconcat() did not support that, therefore they are now all in one row

alt.FacetChart(...)

In [16]:
#Combining the charts for comparison
combined_chart2 = alt.hconcat(
    scatter_2,
    facet_bubble
).configure_concat(
    spacing=50  # Space between the charts
).properties(
    title={
        "text": ("How does elevation gain on standard routes correlate with distance, difficulty and visitor traffic?", "Are longer routes necessarily harder and less visited?"),
        "subtitle": ["For more information zoom in or hover over the points", "", ""],
        "fontSize": 18,
        "subtitleFontSize": 14,
        "anchor": 'middle'
    }
)

combined_chart2

alt.HConcatChart(...)

#### **a) Visual Encodings and Interactivity**

**Marks** (identical for both plots): mark_circle, circles represent individual routes to the mountain peaks

**Encoding Channels** (identical for both plots):
- Magnitude Channels (Quantitative):
    - X: Route Distance, position on a common scale (Distance_mi:Q)
    - Y: Elevation Gain, position on a common scale (Elevation Gain_ft:Q)
    - Size: circle size represents visitor count per route (Average Visitors 2017:Q)
- Identity Channels (Nominal):
    - Color: Difficulty, color hue indicates trail difficulty (Difficulty:N)

**Interactivity**
- .interactive() enables zoom and interactive activity within the map
- Tooltip displaying detailed values for Distance_mi, Elevation Gain_ft, Average Visitors, and Difficulty.

**Faceted Bubble Chart** has separate panels for each difficulty level, enabling route comparison across the individual trail difficulties.



#### **b) Strongest Visualization and Rationale**
**Scatter Plot:** The scatter plot is best for comparing multiple variables at once - such as distance, elevation gain, difficulty, and visitor numbers - in a single view. It allows for cross-category comparisons (e.g., seeing if harder hikes have longer distances or more elevation gain).


**Faceted Bubble Chart:** This chart is best for examining trends within each difficulty level independently. It avoids overplotting and makes it easier to see patterns within a single category (e.g., how visitor numbers vary among "easy" hikes).


**Final Conclusion:** In my opinion, the scatter plot is the most effective choice for answering the initial question about how elevation gain relates to distance, difficulty, and visitor numbers. It brings all the key variables together in one view, making it easy to spot patterns and correlations. Plus, with the interactive feature and the tooltip, it’s simple to explore the data, even if you’re not an expert.



#### **c) Gained insights**
**Scatter Plot:**
- Correlation between distance and elevation gain:
    - there is definitely a trend visible, the longer the route the more elevation gain there is
- Visitor Popularity:
    - larger bubbles (more visitors) are mostly clustered around easy to moderate-difficulty hikes, with less elevation gain


**Faceted Bubble Chart:**
- Examining the Difficulty:
    - hikes classified as easy cluster at lower distances and elevation gain
    - most hikes are classified as medium, there's only a few easy ones and a few hard ones
- Insights into Visitor distribution:
    - easy to moderate hikes attract the most visitors (see largest bubbles)
    - longer hiking distances do not seem to impact the visitor numbers as much as the difficulty of the hike

### **Question 3:** <br>

**Mountain range elevation comparison**

* **Charts:** Bar Chart, Boxplot
* **Insights:** Sawatch Range has highest average elevation; variability in other ranges, comparison of distributions

In [17]:
#Question 3 | Chart 1: Bar Chart with Selection
selection = alt.selection_point()

bar_chart = alt.Chart(df_col14er).mark_bar().encode(
    y=alt.Y('Mountain Range:N', 
           title='Mountain Range',
           sort=alt.EncodingSortField('Elevation_ft', op='mean', order='descending'), #sorting by the mean
           axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    x=alt.X('mean(Elevation_ft):Q',     #explicit mean aggregation needed
           title='Average Elevation (feet)',
           scale=alt.Scale(domain=[14000, 14500]),
           axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    color=alt.Color('Mountain Range:N', legend=None),  #remove legend since ranges are on axis
    tooltip=[
        'Mountain Range:N',
        alt.Tooltip('mean(Elevation_ft):Q', title='Avg Elevation', format='.2f'),
        alt.Tooltip('count()', title='Number of 14ers')
    ],
    opacity=alt.condition(selection, alt.value(0.9), alt.value(0.1))
).properties(
    width=600,  #wider for better readability
    height=300,  #taller
    title={
        "text": "Average Elevation of each Mountain Range",
        "subtitle": "Click on a bar to highlight the corresponding Mountain Range",
        "fontSize": 16,
        "subtitleFontSize": 12,
    }
).add_params(
    selection
)

bar_chart

alt.Chart(...)

In [18]:
#value verification
df_col14er.groupby('Mountain Range')['Elevation_ft'].mean().sort_values(ascending=False).round(2)

Mountain Range
Sawatch Range             14216.00
Front Range               14210.50
Mosquito Range            14188.00
Elk Mountains             14140.60
Sangre de Cristo Range    14137.40
San Juan Mountains        14095.92
Name: Elevation_ft, dtype: float64

In [19]:
#Question 3 | Chart 2: Boxplot with Selection

#selection for mountain range
selection1 = alt.selection_point(fields=['Mountain Range'])

#compute median elevation per mountain range and sort descending
sort_order = (
    df_col14er.groupby("Mountain Range")["Elevation_ft"]
    .mean()
    .sort_values(ascending=False)
    .index
    .tolist()
)


#base boxplot (static)
box_base = alt.Chart(df_col14er).mark_boxplot(size=25, extent=1.5, opacity=0.5).encode(
    y=alt.Y('Mountain Range:N',
            sort=sort_order,
            title='Mountain Range',
            axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    x=alt.X('Elevation_ft:Q',
            title='Elevation Distribution (feet)',
            scale=alt.Scale(domain=[14000, 14500]),
            axis=alt.Axis(labelFontSize=12, titleFontSize=14)),
    color=alt.Color('Mountain Range:N', legend=None),  # Set opacity for the boxplot
).properties(
    width=600,
    height=300,
    title={
        "text": "Elevation Distribution of Colorado 14ers by Mountain Range",
        "subtitle": "Click on a boxplot to highlight the corresponding Mountain Range and Mountain Peaks",
        "fontSize": 16,
        "subtitleFontSize": 12,
    }
)

#add jittered points on top with opacity controlled by selection
points = alt.Chart(df_col14er).mark_circle(size=60).encode(
    y=alt.Y('Mountain Range:N', sort=sort_order),
    x='Elevation_ft:Q',
    opacity=alt.condition(selection1, alt.value(0.9), alt.value(0.1)),
    tooltip=['Mountain Peak', 'Elevation_ft', 'Mountain Range'],
    color=alt.Color('Mountain Range:N', legend=None)
).add_params(
    selection1
)

#layer both
boxplot = box_base + points
boxplot


alt.LayerChart(...)

In [20]:
#Combinig the charts for comparison
combined_chart3 = alt.hconcat(
    bar_chart,
    boxplot
).configure_concat(
    spacing=50  # Space between the charts
).properties(
    title={
        "text": "Which mountain range has the highest average elevation, and how do other ranges compare?",
        "subtitle": ["For more information click on the bars/boxplots or hover over them", "", ""],
        "fontSize": 18,
        "subtitleFontSize": 14,
        "anchor":'middle'
    }
)

combined_chart3

alt.HConcatChart(...)

#### **a) Visual Encodings and Interactivity**

**Bar Chart:**
**Marks:**
- Bar Chart: mark_bar(), rectangular bars encoding mean elevation per mountain range
- Boxplot: mark_boxplot(), boxes, whiskers, outliers represent the distribution of elevation;  includes an additional layer of individual Mountain Peaks as points to support interactivity

**Encoding Channels for Bar Chart:**
- Magnitude Channels (Quantitative) for Bar Chart:
    - X: mean, position on a common scale (average elevation)
    - Y: Mountain Range:N, categorical grouping by range (sorted by mean elevation)
- Magnitude Channels (Quantitative) for Boxplot:
    - X: Elevation_ft:Q, position on a common scale (full distribution: min, Q1, median, Q3, max)
    - Y: Mountain Range:N, categorical grouping (sorted by mean elevation)
- Identity Channels (Nominal):
    - Color: Mountain Range:N, color hue reinforces categorical distinction (redundant with Y)


**Interactivity**
- Bar Chart Tooltip: Shows mean of elevation and number of peaks for each range (quantitative + nominal)
- Boxplot Tooltip: Shows detailed elevation stats: min, Q1, median, Q3, and max for each range; interactive points display individual mountain peak names and exact elevation values on hover
- Selection of a bar/boxplot highlights the individual Mountain Range



#### **b) Strongest Visualization and Rationale**
**Bar Chart:**  The bar chart is ideal for answering this question because:
1. Direct Mean Comparison: The bars explicitly encode mean elevation via x=alt.X('mean(Elevation_ft):Q'), allowing immediate comparison of average elevations.
2. Sorting: Ranges are ordered by mean elevation (descending), making the ranking very obvious and easy to understand.
3. Precision: The tooltip provides the exact mean values (e.g., "14216.00 ft") eliminating guessing, since one can easily check the value by hovering over the bars.

**Why not the Boxplot?** 
- The boxplot encodes the median and not the mean elevation.
- While it shows distributions, the question focuses on averages, which are less intuitive to interpret compared to bar charts. Therefore in this case they are less useful.



#### **c) Gained insights**
**Bar Chart:**
- Sawatch Range has the highest average elevation (14,216 ft) and the highest number of Mountain Peaks (15).
- Front Range is close in second place (14,210.5 ft).
- San Juan Mountains are lowest (14,095.9 ft), with others in between (e.g., Mosquito Range: 14,188 ft).

**Box Plot:**
- Sawatch Range has the widest elevation spread (longest whiskers), indicating more variability in peak heights
- Several ranges (like San Juan Mountains) have outliers - individual peaks that deviate significantly from their range's typical elevation

## 8. Exported Visualizations

These `.html` files are also included in the repository for direct viewing outside the notebook.

In [21]:
scatter_1.save('Q1_scatter_1.html')
scatter_2.save('Q2_scatter_2.html')
bar_chart.save('Q3_bar_chart.html')

## 9. Conclusion

This project showed how data processing and interactive visualizations can reveal meaningful insights about Colorado's highest mountain peaks.
- **Geospatial analysis** revealed clusters and elevation differences by mountain range
- **Correlation analysis** showed clear links between distance, elevation gain, difficulty and visitor popularity
- **Comparisons across ranges** highlighted Sawatch Range as the highest on average
<br>

Future improvements could include adding topographic basemaps, visitor trend data over time or incorporating weather/climate data to enrich the analysis.